# Portfolio Return Construction — HPG, FPT, MWG

## Objective

This notebook formalizes individual-stock return calculations and constructs the default equal-weight portfolio return used in the Value at Risk pipeline.

Both simple returns and log returns are retained because they serve different purposes:

- simple returns are used for cross-sectional portfolio aggregation;
- log returns are useful for time-series analysis and statistical interpretation.

The default portfolio contains HPG, FPT, and MWG with equal constant weights.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


# Project configuration

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

DATA_DIR = PROJECT_ROOT / "data" / "processed"

TICKERS = [
    "HPG",
    "FPT",
    "MWG",
]

EXPECTED_COLUMNS = [
    "date",
    "ticker",
    "open",
    "high",
    "low",
    "close",
    "volume",
]


# Load validated datasets

frames = []

for ticker in TICKERS:
    file_path = DATA_DIR / f"{ticker}_clean.csv"

    data = pd.read_csv(
        file_path,
        parse_dates=["date"],
    )

    if data.columns.tolist() != EXPECTED_COLUMNS:
        raise ValueError(
            f"Unexpected schema for {ticker}: "
            f"{data.columns.tolist()}"
        )

    frames.append(data)


market_data = (
    pd.concat(
        frames,
        ignore_index=True,
    )
    .sort_values(
        ["ticker", "date"]
    )
    .reset_index(drop=True)
)


# Calculate individual-stock returns

market_data["previous_close"] = (
    market_data
    .groupby("ticker")["close"]
    .shift(1)
)

market_data["simple_return"] = (
    market_data["close"]
    / market_data["previous_close"]
    - 1
)

market_data["log_return"] = np.log(
    market_data["close"]
    / market_data["previous_close"]
)


# Return calculation sanity check

return_check = (
    market_data
    .groupby("ticker")
    .agg(
        observations=(
            "date",
            "size",
        ),
        valid_simple_returns=(
            "simple_return",
            "count",
        ),
        missing_simple_returns=(
            "simple_return",
            lambda series: series.isna().sum(),
        ),
        valid_log_returns=(
            "log_return",
            "count",
        ),
        missing_log_returns=(
            "log_return",
            lambda series: series.isna().sum(),
        ),
    )
)


print(
    f"Combined market data shape: {market_data.shape}"
)

return_check

Combined market data shape: (4914, 10)


,observations,valid_simple_returns,missing_simple_returns,valid_log_returns,missing_log_returns
ticker,,,,,
FPT,1638,1637,1,1637,1
HPG,1638,1637,1,1637,1
MWG,1638,1637,1,1637,1


In [2]:
# Validate simple-return and log-return identity

RETURN_TOLERANCE = 1e-12

market_data["implied_log_return"] = np.log1p(
    market_data["simple_return"]
)

market_data["log_return_difference"] = (
    market_data["log_return"]
    - market_data["implied_log_return"]
)


return_identity_check = (
    market_data
    .groupby("ticker")
    .agg(
        valid_comparisons=(
            "log_return_difference",
            "count",
        ),
        max_absolute_difference=(
            "log_return_difference",
            lambda series: series.abs().max(),
        ),
        mean_absolute_difference=(
            "log_return_difference",
            lambda series: series.abs().mean(),
        ),
    )
)


identity_violations = market_data.loc[
    market_data["log_return_difference"].abs()
    > RETURN_TOLERANCE,
    [
        "date",
        "ticker",
        "simple_return",
        "log_return",
        "implied_log_return",
        "log_return_difference",
    ],
]


print(
    f"Return identity tolerance: {RETURN_TOLERANCE:.0e}"
)

display(return_identity_check)

print(
    "Rows exceeding tolerance:",
    len(identity_violations),
)


market_data[
    [
        "date",
        "ticker",
        "close",
        "previous_close",
        "simple_return",
        "log_return",
        "implied_log_return",
    ]
].head(9)

Return identity tolerance: 1e-12


,valid_comparisons,max_absolute_difference,mean_absolute_difference
ticker,,,
FPT,1637,0.0,0.0
HPG,1637,0.0,0.0
MWG,1637,0.0,0.0


Rows exceeding tolerance: 0


,date,ticker,close,previous_close,simple_return,log_return,implied_log_return
0,2020-01-02,FPT,20.90,NaN,NaN,NaN,NaN
1,2020-01-03,FPT,20.54,20.90,-0.017225,-0.017375,-0.017375
2,2020-01-06,FPT,20.33,20.54,-0.010224,-0.010277,-0.010277
3,2020-01-07,FPT,20.72,20.33,0.019183,0.019002,0.019002
4,2020-01-08,FPT,20.26,20.72,-0.022201,-0.022451,-0.022451
5,2020-01-09,FPT,20.54,20.26,0.013820,0.013726,0.013726
6,2020-01-10,FPT,20.50,20.54,-0.001947,-0.001949,-0.001949
7,2020-01-13,FPT,20.36,20.50,-0.006829,-0.006853,-0.006853
8,2020-01-14,FPT,20.33,20.36,-0.001473,-0.001475,-0.001475


In [3]:
# Validate return quality before portfolio aggregation

return_quality_check = (
    market_data
    .groupby("ticker")
    .agg(
        observations=(
            "date",
            "size",
        ),
        valid_simple_returns=(
            "simple_return",
            "count",
        ),
        missing_simple_returns=(
            "simple_return",
            lambda series: series.isna().sum(),
        ),
        positive_inf_simple_returns=(
            "simple_return",
            lambda series: np.isposinf(series).sum(),
        ),
        negative_inf_simple_returns=(
            "simple_return",
            lambda series: np.isneginf(series).sum(),
        ),
        simple_return_min=(
            "simple_return",
            "min",
        ),
        simple_return_max=(
            "simple_return",
            "max",
        ),
        valid_log_returns=(
            "log_return",
            "count",
        ),
        missing_log_returns=(
            "log_return",
            lambda series: series.isna().sum(),
        ),
        positive_inf_log_returns=(
            "log_return",
            lambda series: np.isposinf(series).sum(),
        ),
        negative_inf_log_returns=(
            "log_return",
            lambda series: np.isneginf(series).sum(),
        ),
        log_return_min=(
            "log_return",
            "min",
        ),
        log_return_max=(
            "log_return",
            "max",
        ),
        invalid_log_domain_count=(
            "simple_return",
            lambda series: (series <= -1).sum(),
        ),
    )
)


return_quality_display = (
    return_quality_check
    .rename(
        columns={
            "simple_return_min": "simple_min_pct",
            "simple_return_max": "simple_max_pct",
            "log_return_min": "log_min_pct",
            "log_return_max": "log_max_pct",
        }
    )
    .copy()
)


percentage_columns = [
    "simple_min_pct",
    "simple_max_pct",
    "log_min_pct",
    "log_max_pct",
]

return_quality_display[
    percentage_columns
] = (
    return_quality_display[
        percentage_columns
    ]
    * 100
)


return_quality_display

,observations,valid_simple_returns,missing_simple_returns,positive_inf_simple_returns,negative_inf_simple_returns,simple_min_pct,simple_max_pct,valid_log_returns,missing_log_returns,positive_inf_log_returns,negative_inf_log_returns,log_min_pct,log_max_pct,invalid_log_domain_count
ticker,,,,,,,,,,,,,,
FPT,1638,1637,1,0,0,-6.989699,7.000000,1637,1,0,0,-7.245994,6.765865,0
HPG,1638,1637,1,0,0,-7.008885,6.944444,1637,1,0,0,-7.266623,6.713930,0
MWG,1638,1637,1,0,0,-7.002918,6.995885,1637,1,0,0,-7.260207,6.762019,0


In [4]:
# Build date-aligned simple-return matrix

duplicate_date_ticker_pairs = (
    market_data
    .duplicated(
        subset=["date", "ticker"],
        keep=False,
    )
    .sum()
)

if duplicate_date_ticker_pairs > 0:
    raise ValueError(
        "Duplicate date-ticker observations detected: "
        f"{duplicate_date_ticker_pairs}"
    )


simple_return_matrix = (
    market_data
    .pivot(
        index="date",
        columns="ticker",
        values="simple_return",
    )
    .sort_index()
    .reindex(columns=TICKERS)
)


complete_return_rows = (
    simple_return_matrix
    .notna()
    .all(axis=1)
    .sum()
)

incomplete_return_rows = (
    simple_return_matrix
    .isna()
    .any(axis=1)
    .sum()
)


return_matrix_check = pd.Series(
    {
        "number_of_dates": len(simple_return_matrix),
        "start_date": simple_return_matrix.index.min(),
        "end_date": simple_return_matrix.index.max(),
        "complete_return_rows": complete_return_rows,
        "incomplete_return_rows": incomplete_return_rows,
    },
    name="value",
)


missing_returns_by_ticker = (
    simple_return_matrix
    .isna()
    .sum()
    .rename("missing_returns")
)


print(
    "Duplicate date-ticker pairs:",
    duplicate_date_ticker_pairs,
)

display(return_matrix_check)

display(missing_returns_by_ticker)

simple_return_matrix.head()

Duplicate date-ticker pairs: 0


number_of_dates                          1638
start_date                2020-01-02 00:00:00
end_date                  2026-07-28 00:00:00
complete_return_rows                     1637
incomplete_return_rows                      1
Name: value, dtype: object

ticker
HPG    1
FPT    1
MWG    1
Name: missing_returns, dtype: int64

ticker,HPG,FPT,MWG
date,,,
2020-01-02,NaN,NaN,NaN
2020-01-03,0.006775,-0.017225,-0.014580
2020-01-06,-0.006729,-0.010224,-0.005025
2020-01-07,-0.012195,0.019183,0.007856
2020-01-08,-0.010974,-0.022201,-0.024220


In [5]:
# Define and validate portfolio weights

WEIGHT_TOLERANCE = 1e-12

weights = pd.Series(
    data=1.0 / len(TICKERS),
    index=TICKERS,
    name="weight",
    dtype=float,
)


 
# Validate asset alignment
 

if weights.index.tolist() != simple_return_matrix.columns.tolist():
    raise ValueError(
        "Portfolio weights are not aligned with return-matrix columns. "
        f"Weights: {weights.index.tolist()}, "
        f"returns: {simple_return_matrix.columns.tolist()}"
    )


 
# Validate weight constraints
 

has_negative_weight = (weights < 0).any()

if has_negative_weight:
    raise ValueError(
        "Negative portfolio weights are not allowed "
        "for the current long-only portfolio."
    )


total_weight = weights.sum()

weights_sum_to_one = np.isclose(
    total_weight,
    1.0,
    atol=WEIGHT_TOLERANCE,
    rtol=0.0,
)

if not weights_sum_to_one:
    raise ValueError(
        "Portfolio weights must sum to 1. "
        f"Current total weight: {total_weight:.16f}"
    )


 
# Weight sanity-check summary
 

weight_check = pd.Series(
    {
        "number_of_assets": len(weights),
        "minimum_weight": weights.min(),
        "maximum_weight": weights.max(),
        "total_weight": total_weight,
        "has_negative_weight": has_negative_weight,
        "weights_sum_to_one": weights_sum_to_one,
    },
    name="value",
)


display(weights.to_frame())

display(weight_check)

,weight
HPG,0.333333
FPT,0.333333
MWG,0.333333


number_of_assets              3
minimum_weight         0.333333
maximum_weight         0.333333
total_weight                1.0
has_negative_weight       False
weights_sum_to_one         True
Name: value, dtype: object

In [6]:
 
# Calculate and validate equal-weight portfolio simple return
 

PORTFOLIO_RETURN_TOLERANCE = 1e-12


 
# Method 1: matrix multiplication
 

portfolio_simple_return = (
    simple_return_matrix
    .dot(weights)
    .rename("portfolio_simple_return")
)


 
# Method 2: explicit weighted-return formula
 

portfolio_simple_return_explicit = (
    simple_return_matrix["HPG"] * weights["HPG"]
    + simple_return_matrix["FPT"] * weights["FPT"]
    + simple_return_matrix["MWG"] * weights["MWG"]
).rename("portfolio_simple_return_explicit")


 
# Compare both implementations
 

portfolio_return_difference = (
    portfolio_simple_return
    - portfolio_simple_return_explicit
).rename("portfolio_return_difference")


portfolio_return_violations = (
    portfolio_return_difference
    .loc[
        portfolio_return_difference.abs()
        > PORTFOLIO_RETURN_TOLERANCE
    ]
)


 
# Portfolio-return sanity-check summary
 

portfolio_return_check = pd.Series(
    {
        "number_of_dates": len(portfolio_simple_return),
        "valid_portfolio_returns": portfolio_simple_return.count(),
        "missing_portfolio_returns": portfolio_simple_return.isna().sum(),
        "portfolio_min_return": portfolio_simple_return.min(),
        "portfolio_max_return": portfolio_simple_return.max(),
        "max_absolute_difference": portfolio_return_difference.abs().max(),
        "mean_absolute_difference": portfolio_return_difference.abs().mean(),
        "rows_exceeding_tolerance": len(portfolio_return_violations),
    },
    name="value",
)


portfolio_return_check_display = (
    portfolio_return_check
    .rename(
        index={
            "portfolio_min_return": "portfolio_min_pct",
            "portfolio_max_return": "portfolio_max_pct",
        }
    )
    .copy()
)

portfolio_return_check_display.loc[
    [
        "portfolio_min_pct",
        "portfolio_max_pct",
    ]
] *= 100


 
# Preview asset and portfolio returns
 

portfolio_return_preview = (
    simple_return_matrix
    .copy()
    .assign(
        portfolio_simple_return=portfolio_simple_return,
        portfolio_simple_return_explicit=portfolio_simple_return_explicit,
        portfolio_return_difference=portfolio_return_difference,
    )
)


display(portfolio_return_check_display)

portfolio_return_preview.head(8)

number_of_dates              1638.000000
valid_portfolio_returns      1637.000000
missing_portfolio_returns       1.000000
portfolio_min_pct              -6.983986
portfolio_max_pct               6.887777
max_absolute_difference         0.000000
mean_absolute_difference        0.000000
rows_exceeding_tolerance        0.000000
Name: value, dtype: float64

ticker,HPG,FPT,MWG,portfolio_simple_return,portfolio_simple_return_explicit,portfolio_return_difference
date,,,,,,
2020-01-02,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-03,0.006775,-0.017225,-0.014580,-0.008343,-0.008343,0.0
2020-01-06,-0.006729,-0.010224,-0.005025,-0.007326,-0.007326,0.0
2020-01-07,-0.012195,0.019183,0.007856,0.004948,0.004948,0.0
2020-01-08,-0.010974,-0.022201,-0.024220,-0.019132,-0.019132,0.0
2020-01-09,0.023578,0.013820,0.015121,0.017507,0.017507,0.0
2020-01-10,0.008130,-0.001947,0.005059,0.003747,0.003747,0.0
2020-01-13,0.002688,-0.006829,-0.005034,-0.003058,-0.003058,0.0


In [7]:
 
# Compare portfolio log return with weighted individual log returns
 

LOG_AGGREGATION_TOLERANCE = 1e-15


 
# Exact log return of the constructed portfolio
 

portfolio_log_return = np.log1p(
    portfolio_simple_return
).rename("portfolio_log_return")


 
# Build date-aligned individual log-return matrix
 

individual_log_return_matrix = (
    market_data
    .pivot(
        index="date",
        columns="ticker",
        values="log_return",
    )
    .sort_index()
    .reindex(columns=TICKERS)
)


 
# Weighted average of individual log returns
 

weighted_average_individual_log_return = (
    individual_log_return_matrix
    .dot(weights)
    .rename("weighted_average_individual_log_return")
)


 
# Compare the two quantities
 

log_aggregation_difference = (
    portfolio_log_return
    - weighted_average_individual_log_return
).rename("log_aggregation_difference")


rows_with_nonzero_difference = (
    log_aggregation_difference
    .abs()
    .gt(LOG_AGGREGATION_TOLERANCE)
    .sum()
)


log_aggregation_check = pd.Series(
    {
        "number_of_dates": len(portfolio_log_return),
        "valid_portfolio_log_returns": portfolio_log_return.count(),
        "missing_portfolio_log_returns": portfolio_log_return.isna().sum(),
        "max_absolute_difference": log_aggregation_difference.abs().max(),
        "mean_absolute_difference": log_aggregation_difference.abs().mean(),
        "rows_with_nonzero_difference": rows_with_nonzero_difference,
    },
    name="value",
)


comparison_preview = pd.concat(
    [
        portfolio_simple_return,
        portfolio_log_return,
        weighted_average_individual_log_return,
        log_aggregation_difference,
    ],
    axis=1,
)


display(log_aggregation_check)

comparison_preview.head(8)

number_of_dates                  1638.000000
valid_portfolio_log_returns      1637.000000
missing_portfolio_log_returns       1.000000
max_absolute_difference             0.001247
mean_absolute_difference            0.000070
rows_with_nonzero_difference     1636.000000
Name: value, dtype: float64

,portfolio_simple_return,portfolio_log_return,weighted_average_individual_log_return,log_aggregation_difference
date,,,,
2020-01-02,NaN,NaN,NaN,NaN
2020-01-03,-0.008343,-0.008378,-0.008437,0.000058
2020-01-06,-0.007326,-0.007353,-0.007356,0.000002
2020-01-07,0.004948,0.004936,0.004852,0.000084
2020-01-08,-0.019132,-0.019317,-0.019335,0.000018
2020-01-09,0.017507,0.017355,0.017346,0.000009
2020-01-10,0.003747,0.003740,0.003731,0.000009
2020-01-13,-0.003058,-0.003063,-0.003071,0.000009


In [8]:
# Validate the Jensen gap in portfolio log-return aggregation

JENSEN_TOLERANCE = 1e-15

valid_jensen_gap = (
    log_aggregation_difference
    .dropna()
)


negative_gap_violations = (
    valid_jensen_gap
    .lt(-JENSEN_TOLERANCE)
    .sum()
)

near_zero_gap_mask = (
    valid_jensen_gap
    .abs()
    .le(JENSEN_TOLERANCE)
)

near_zero_gap_rows = (
    near_zero_gap_mask
    .sum()
)

maximum_gap_date = (
    valid_jensen_gap
    .idxmax()
)


jensen_check = pd.Series(
    {
        "valid_comparisons": len(valid_jensen_gap),
        "minimum_jensen_gap": valid_jensen_gap.min(),
        "maximum_jensen_gap": valid_jensen_gap.max(),
        "mean_jensen_gap": valid_jensen_gap.mean(),
        "negative_gap_violations": negative_gap_violations,
        "near_zero_gap_rows": near_zero_gap_rows,
        "maximum_gap_date": maximum_gap_date,
    },
    name="value",
)


near_zero_dates = (
    valid_jensen_gap
    .index[near_zero_gap_mask]
)


near_zero_gap_preview = (
    simple_return_matrix
    .loc[near_zero_dates]
    .copy()
    .assign(
        portfolio_simple_return=portfolio_simple_return.loc[
            near_zero_dates
        ],
        portfolio_log_return=portfolio_log_return.loc[
            near_zero_dates
        ],
        weighted_average_individual_log_return=(
            weighted_average_individual_log_return.loc[
                near_zero_dates
            ]
        ),
        log_aggregation_difference=(
            log_aggregation_difference.loc[
                near_zero_dates
            ]
        ),
    )
)


display(jensen_check)

near_zero_gap_preview

valid_comparisons                         1637
minimum_jensen_gap                         0.0
maximum_jensen_gap                    0.001247
mean_jensen_gap                        0.00007
negative_gap_violations                      0
near_zero_gap_rows                           1
maximum_gap_date           2023-11-01 00:00:00
Name: value, dtype: object

ticker,HPG,FPT,MWG,portfolio_simple_return,portfolio_log_return,weighted_average_individual_log_return,log_aggregation_difference
date,,,,,,,
2025-12-02,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [9]:
# Build canonical portfolio-return dataset

portfolio_returns = (
    simple_return_matrix
    .rename(
        columns={
            "HPG": "HPG_simple_return",
            "FPT": "FPT_simple_return",
            "MWG": "MWG_simple_return",
        }
    )
    .assign(
        portfolio_simple_return=portfolio_simple_return,
        portfolio_log_return=portfolio_log_return,
    )
    .reset_index()
    .dropna(
        subset=["portfolio_simple_return"]
    )
    .sort_values("date")
    .reset_index(drop=True)
)


 
# Validate canonical dataset
 

duplicate_portfolio_dates = (
    portfolio_returns["date"]
    .duplicated()
    .sum()
)

if duplicate_portfolio_dates > 0:
    raise ValueError(
        "Duplicate dates detected in portfolio-return dataset: "
        f"{duplicate_portfolio_dates}"
    )


portfolio_dataset_check = pd.Series(
    {
        "observations": len(portfolio_returns),
        "start_date": portfolio_returns["date"].min(),
        "end_date": portfolio_returns["date"].max(),
        "duplicate_dates": duplicate_portfolio_dates,
        "missing_values": portfolio_returns.isna().sum().sum(),
        "minimum_portfolio_return": (
            portfolio_returns["portfolio_simple_return"].min()
        ),
        "maximum_portfolio_return": (
            portfolio_returns["portfolio_simple_return"].max()
        ),
    },
    name="value",
)


portfolio_dataset_check_display = (
    portfolio_dataset_check
    .rename(
        index={
            "minimum_portfolio_return": (
                "minimum_portfolio_return_pct"
            ),
            "maximum_portfolio_return": (
                "maximum_portfolio_return_pct"
            ),
        }
    )
    .copy()
)


portfolio_dataset_check_display.loc[
    [
        "minimum_portfolio_return_pct",
        "maximum_portfolio_return_pct",
    ]
] *= 100


display(portfolio_dataset_check_display)

display(portfolio_returns.head(10))

portfolio_returns.tail(5)

observations                                   1637
start_date                      2020-01-03 00:00:00
end_date                        2026-07-28 00:00:00
duplicate_dates                                   0
missing_values                                    0
minimum_portfolio_return_pct              -6.983986
maximum_portfolio_return_pct               6.887777
Name: value, dtype: object

ticker,date,HPG_simple_return,FPT_simple_return,MWG_simple_return,portfolio_simple_return,portfolio_log_return
0,2020-01-03,0.006775,-0.017225,-0.014580,-0.008343,-0.008378
1,2020-01-06,-0.006729,-0.010224,-0.005025,-0.007326,-0.007353
2,2020-01-07,-0.012195,0.019183,0.007856,0.004948,0.004936
3,2020-01-08,-0.010974,-0.022201,-0.024220,-0.019132,-0.019317
4,2020-01-09,0.023578,0.013820,0.015121,0.017507,0.017355
5,2020-01-10,0.008130,-0.001947,0.005059,0.003747,0.003740
6,2020-01-13,0.002688,-0.006829,-0.005034,-0.003058,-0.003063
7,2020-01-14,0.030831,-0.001473,-0.005340,0.008006,0.007974
8,2020-01-15,0.005202,0.000000,0.000848,0.002016,0.002014
9,2020-01-16,0.006468,0.017216,0.012140,0.011941,0.011871


ticker,date,HPG_simple_return,FPT_simple_return,MWG_simple_return,portfolio_simple_return,portfolio_log_return
1632,2026-07-22,-0.004808,-0.003086,-0.069842,-0.025912,-0.026254
1633,2026-07-23,0.004831,0.001548,0.012658,0.006346,0.006326
1634,2026-07-24,0.000000,-0.027821,-0.048864,-0.025561,-0.025894
1635,2026-07-27,-0.021635,-0.011129,-0.069146,-0.033970,-0.034560
1636,2026-07-28,0.031941,0.012862,0.030002,0.024935,0.024629


In [ ]:
 
# Export canonical portfolio returns and review sample
 

portfolio_returns_export = portfolio_returns.copy()
portfolio_returns_export.columns.name = None


 
# Output paths
 

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
SAMPLE_DIR = PROJECT_ROOT / "data" / "sample"

SAMPLE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

portfolio_returns_path = (
    PROCESSED_DIR / "portfolio_returns.csv"
)

portfolio_return_sample_path = (
    SAMPLE_DIR / "portfolio_returns_sample.csv"
)


 
# Construct representative sample
 

sample_size_per_section = 5
middle_start = (
    len(portfolio_returns_export) // 2
    - sample_size_per_section // 2
)

portfolio_return_sample = (
    pd.concat(
        [
            portfolio_returns_export.head(
                sample_size_per_section
            ),
            portfolio_returns_export.iloc[
                middle_start:
                middle_start + sample_size_per_section
            ],
            portfolio_returns_export.tail(
                sample_size_per_section
            ),
        ],
        ignore_index=True,
    )
    .drop_duplicates(
        subset=["date"]
    )
    .sort_values("date")
    .reset_index(drop=True)
)


 
# Export
 

portfolio_returns_export.to_csv(
    portfolio_returns_path,
    index=False,
)

portfolio_return_sample.to_csv(
    portfolio_return_sample_path,
    index=False,
)


 
# Read-back validation
 

portfolio_returns_reloaded = pd.read_csv(
    portfolio_returns_path,
    parse_dates=["date"],
)

portfolio_return_sample_reloaded = pd.read_csv(
    portfolio_return_sample_path,
    parse_dates=["date"],
)


pd.testing.assert_frame_equal(
    portfolio_returns_export.reset_index(drop=True),
    portfolio_returns_reloaded,
    check_dtype=False,
    rtol=1e-12,
    atol=1e-12,
)


print("Round-trip validation: PASS")


 
# Export sanity-check summary
 

expected_export_columns = (
    portfolio_returns_export
    .columns
    .tolist()
)


export_check = pd.Series(
    {
        "full_rows": len(portfolio_returns_reloaded),
        "sample_rows": len(portfolio_return_sample_reloaded),
        "full_columns_match": (
            portfolio_returns_reloaded.columns.tolist()
            == expected_export_columns
        ),
        "sample_columns_match": (
            portfolio_return_sample_reloaded.columns.tolist()
            == expected_export_columns
        ),
        "full_missing_values": (
            portfolio_returns_reloaded
            .isna()
            .sum()
            .sum()
        ),
        "sample_missing_values": (
            portfolio_return_sample_reloaded
            .isna()
            .sum()
            .sum()
        ),
        "full_duplicate_dates": (
            portfolio_returns_reloaded["date"]
            .duplicated()
            .sum()
        ),
        "sample_duplicate_dates": (
            portfolio_return_sample_reloaded["date"]
            .duplicated()
            .sum()
        ),
    },
    name="value",
)


display(export_check)

portfolio_return_sample_reloaded

Round-trip validation: PASS


full_rows                 1637
sample_rows                 15
full_columns_match        True
sample_columns_match      True
full_missing_values          0
sample_missing_values        0
full_duplicate_dates         0
sample_duplicate_dates       0
Name: value, dtype: object

,date,HPG_simple_return,FPT_simple_return,MWG_simple_return,portfolio_simple_return,portfolio_log_return
0,2020-01-03,0.006775,-0.017225,-0.014580,-0.008343,-0.008378
1,2020-01-06,-0.006729,-0.010224,-0.005025,-0.007326,-0.007353
2,2020-01-07,-0.012195,0.019183,0.007856,0.004948,0.004936
3,2020-01-08,-0.010974,-0.022201,-0.024220,-0.019132,-0.019317
4,2020-01-09,0.023578,0.013820,0.015121,0.017507,0.017355
5,2023-04-12,-0.002135,0.000000,0.001288,-0.000282,-0.000282
6,2023-04-13,-0.014265,-0.008727,0.007461,-0.005177,-0.005191
7,2023-04-14,0.009407,0.000000,-0.025536,-0.005377,-0.005391
8,2023-04-17,0.010036,0.000000,-0.003931,0.002035,0.002033
9,2023-04-18,0.004258,-0.002457,0.002631,0.001477,0.001476


In [ ]:
 
# Audit VN-Index raw data before benchmark adoption
 

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

vnindex_path = RAW_DATA_DIR / "VNINDEX.csv"

vnindex_raw = pd.read_csv(vnindex_path)


 
# Inspect raw structure
 

print("VN-Index raw shape:", vnindex_raw.shape)

print(
    "\nVN-Index raw columns:",
    vnindex_raw.columns.tolist(),
)

print("\nVN-Index raw dtypes:")
display(vnindex_raw.dtypes)

print("\nFirst 5 rows:")
display(vnindex_raw.head())

print("\nLast 5 rows:")
display(vnindex_raw.tail())


 
# Resolve and standardize date field for audit only
 

if "date" in vnindex_raw.columns:
    vnindex_date_column = "date"
elif "time" in vnindex_raw.columns:
    vnindex_date_column = "time"
else:
    raise ValueError(
        "VN-Index dataset must contain either "
        "'date' or 'time' column."
    )


vnindex_audit = vnindex_raw.copy()

vnindex_audit[vnindex_date_column] = pd.to_datetime(
    vnindex_audit[vnindex_date_column]
)

if vnindex_date_column == "time":
    vnindex_audit = vnindex_audit.rename(
        columns={"time": "date"}
    )


vnindex_audit = (
    vnindex_audit
    .sort_values("date")
    .reset_index(drop=True)
)


 
# Data-quality audit
 

vnindex_quality_check = pd.Series(
    {
        "observations": len(vnindex_audit),
        "start_date": vnindex_audit["date"].min(),
        "end_date": vnindex_audit["date"].max(),
        "duplicate_dates": (
            vnindex_audit["date"]
            .duplicated()
            .sum()
        ),
        "missing_dates": (
            vnindex_audit["date"]
            .isna()
            .sum()
        ),
        "missing_close": (
            vnindex_audit["close"]
            .isna()
            .sum()
        ),
        "nonpositive_close": (
            vnindex_audit["close"]
            .le(0)
            .sum()
        ),
    },
    name="value",
)


 
# Trading-date alignment audit
 

portfolio_dates = pd.Index(
    simple_return_matrix.index.unique()
)

vnindex_dates = pd.Index(
    vnindex_audit["date"].dropna().unique()
)


common_dates = portfolio_dates.intersection(
    vnindex_dates
)

portfolio_dates_missing_in_vnindex = (
    portfolio_dates.difference(vnindex_dates)
)

vnindex_dates_not_in_portfolio = (
    vnindex_dates.difference(portfolio_dates)
)


vnindex_alignment_check = pd.Series(
    {
        "number_of_portfolio_dates": len(portfolio_dates),
        "number_of_vnindex_dates": len(vnindex_dates),
        "common_dates": len(common_dates),
        "portfolio_dates_missing_in_vnindex": len(
            portfolio_dates_missing_in_vnindex
        ),
        "vnindex_dates_not_in_portfolio": len(
            vnindex_dates_not_in_portfolio
        ),
    },
    name="value",
)


display(vnindex_quality_check)

display(vnindex_alignment_check)

VN-Index raw shape: (1638, 6)

VN-Index raw columns: ['time', 'open', 'high', 'low', 'close', 'volume']

VN-Index raw dtypes:


time       object
open      float64
high      float64
low       float64
close     float64
volume      int64
dtype: object


First 5 rows:


,time,open,high,low,close,volume
0,2020-01-02 07:00:00,960.26,966.67,959.67,966.67,131523160
1,2020-01-03 07:00:00,968.72,970.88,965.14,965.14,149202550
2,2020-01-06 07:00:00,962.79,963.20,955.59,955.79,140978550
3,2020-01-07 07:00:00,955.39,959.46,953.19,958.88,130272030
4,2020-01-08 07:00:00,954.13,955.84,945.28,948.98,179374340



Last 5 rows:


,time,open,high,low,close,volume
1633,2026-07-22 07:00:00,1729.33,1736.91,1668.53,1668.53,847743499
1634,2026-07-23 07:00:00,1664.50,1703.48,1658.35,1699.38,767199044
1635,2026-07-24 07:00:00,1683.23,1696.78,1674.64,1686.11,497795740
1636,2026-07-27 07:00:00,1694.78,1700.69,1669.01,1669.01,542734230
1637,2026-07-28 07:00:00,1653.51,1684.54,1651.20,1680.62,634234923


observations                        1638
start_date           2020-01-02 07:00:00
end_date             2026-07-28 07:00:00
duplicate_dates                        0
missing_dates                          0
missing_close                          0
nonpositive_close                      0
Name: value, dtype: object

number_of_portfolio_dates             1638
number_of_vnindex_dates               1638
common_dates                             0
portfolio_dates_missing_in_vnindex    1638
vnindex_dates_not_in_portfolio        1638
Name: value, dtype: int64

In [ ]:
 
# Recheck VN-Index alignment using normalized trading dates
 

vnindex_date_aligned = vnindex_audit.copy()

vnindex_date_aligned["date"] = (
    vnindex_date_aligned["date"]
    .dt
    .normalize()
)


portfolio_calendar_dates = pd.DatetimeIndex(
    simple_return_matrix.index
).normalize()


 
# Validate normalization result
 

normalized_vnindex_duplicates = (
    vnindex_date_aligned["date"]
    .duplicated()
    .sum()
)

if normalized_vnindex_duplicates > 0:
    raise ValueError(
        "Duplicate VN-Index trading dates detected after "
        f"date normalization: {normalized_vnindex_duplicates}"
    )


vnindex_calendar_dates = pd.DatetimeIndex(
    vnindex_date_aligned["date"]
    .unique()
).sort_values()

portfolio_calendar_dates = (
    portfolio_calendar_dates
    .unique()
    .sort_values()
)


 
# Compare normalized trading calendars
 

common_calendar_dates = (
    portfolio_calendar_dates
    .intersection(vnindex_calendar_dates)
)

portfolio_dates_missing_in_vnindex = (
    portfolio_calendar_dates
    .difference(vnindex_calendar_dates)
)

vnindex_dates_not_in_portfolio = (
    vnindex_calendar_dates
    .difference(portfolio_calendar_dates)
)


dates_fully_aligned = (
    portfolio_calendar_dates.equals(
        vnindex_calendar_dates
    )
)


alignment_after_normalization_check = pd.Series(
    {
        "number_of_portfolio_dates": len(
            portfolio_calendar_dates
        ),
        "number_of_vnindex_dates": len(
            vnindex_calendar_dates
        ),
        "common_dates": len(
            common_calendar_dates
        ),
        "portfolio_dates_missing_in_vnindex": len(
            portfolio_dates_missing_in_vnindex
        ),
        "vnindex_dates_not_in_portfolio": len(
            vnindex_dates_not_in_portfolio
        ),
        "dates_fully_aligned": dates_fully_aligned,
    },
    name="value",
)


display(alignment_after_normalization_check)


if len(portfolio_dates_missing_in_vnindex) > 0:
    print(
        "\nPortfolio dates missing in VN-Index:"
    )
    display(
        portfolio_dates_missing_in_vnindex[:10]
    )


if len(vnindex_dates_not_in_portfolio) > 0:
    print(
        "\nVN-Index dates not in portfolio:"
    )
    display(
        vnindex_dates_not_in_portfolio[:10]
    )

number_of_portfolio_dates             1638
number_of_vnindex_dates               1638
common_dates                          1638
portfolio_dates_missing_in_vnindex       0
vnindex_dates_not_in_portfolio           0
dates_fully_aligned                   True
Name: value, dtype: object

In [ ]:
 
# Idempotently update data dictionary with portfolio-return methodology
 

DATA_DICTIONARY_PATH = (
    PROJECT_ROOT / "docs" / "data-dictionary.md"
)

SECTION_MARKER = "## Portfolio Return Dataset"


if not DATA_DICTIONARY_PATH.exists():
    raise FileNotFoundError(
        f"Data dictionary not found: {DATA_DICTIONARY_PATH}"
    )


existing_dictionary = DATA_DICTIONARY_PATH.read_text(
    encoding="utf-8"
)


portfolio_return_dictionary_section = """
## Portfolio Return Dataset

### Canonical dataset

The canonical portfolio-return dataset is stored at:

`data/processed/portfolio_returns.csv`

A small review sample is stored at:

`data/sample/portfolio_returns_sample.csv`

The canonical dataset contains 1,637 valid daily return observations
from 2020-01-03 through 2026-07-28, with no duplicate dates and no
missing values.

### Portfolio construction

The default portfolio contains HPG, FPT, and MWG.

The portfolio is long-only, fully invested, and equally weighted:

- HPG weight = 1/3
- FPT weight = 1/3
- MWG weight = 1/3

Individual simple return is defined as:

`R_i,t = P_i,t / P_i,t-1 - 1`

The portfolio simple return is constructed cross-sectionally as:

`R_p,t = sum(w_i * R_i,t)`

The portfolio simple return is the primary portfolio return series
used by the Value at Risk pipeline.

Portfolio log return is derived from the constructed portfolio simple
return:

`r_p,t = ln(1 + R_p,t)`

A weighted average of individual asset log returns is not treated as
the exact portfolio log return. It is used only as a methodological
diagnostic and is not included in the canonical portfolio-return
dataset.

### Variable dictionary

| Variable | Type | Unit | Definition | Role |
| --- | --- | --- | --- | --- |
| `date` | datetime | trading date | Trading date associated with the return observation. | Time index |
| `HPG_simple_return` | float | decimal return | Daily simple return of HPG. | Portfolio input |
| `FPT_simple_return` | float | decimal return | Daily simple return of FPT. | Portfolio input |
| `MWG_simple_return` | float | decimal return | Daily simple return of MWG. | Portfolio input |
| `portfolio_simple_return` | float | decimal return | Equal-weight simple return of HPG, FPT, and MWG. | Primary VaR return series |
| `portfolio_log_return` | float | decimal log return | `ln(1 + portfolio_simple_return)`. | Derived analytical series |

### Missing-value convention

The first price observation for each asset has no previous closing
price and therefore has an undefined return.

This initial undefined return is retained during return construction
and validation, but it is excluded from the canonical portfolio-return
dataset.

As a result, 1,638 aligned price dates produce 1,637 valid portfolio
return observations.

A missing return must not be interpreted or filled as a zero return.

### VN-Index benchmark

VN-Index is designated as a market benchmark and is not a constituent
of the default portfolio.

Its portfolio weight is therefore zero.

The current raw benchmark file is:

`data/raw/VNINDEX.csv`

The raw VN-Index dataset contains 1,638 observations covering
2020-01-02 through 2026-07-28.

Its raw timestamp includes a `07:00:00` time component. For daily
trading-calendar comparison, the timestamp is normalized to midnight
without changing the calendar date.

After normalization, the VN-Index and portfolio price calendars contain
1,638 common trading dates with no missing dates in either direction.

`VNINDEX.csv` remains a raw dataset. No `VNINDEX_clean.csv` processed
benchmark dataset has been established at this stage, so raw VN-Index
data must not be silently treated as model-ready processed data.
""".strip()


 
# Append only when the section does not already exist
 

if SECTION_MARKER not in existing_dictionary:
    updated_dictionary = (
        existing_dictionary.rstrip()
        + "\n\n"
        + portfolio_return_dictionary_section
        + "\n"
    )

    DATA_DICTIONARY_PATH.write_text(
        updated_dictionary,
        encoding="utf-8",
    )

    data_dictionary_status = "CREATED"

else:
    data_dictionary_status = "ALREADY EXISTS"


 
# Post-run validation
 

dictionary_after_run = DATA_DICTIONARY_PATH.read_text(
    encoding="utf-8"
)

marker_count = dictionary_after_run.count(
    SECTION_MARKER
)


if marker_count != 1:
    raise ValueError(
        "Unexpected Portfolio Return Dataset marker count: "
        f"{marker_count}"
    )


print(
    "Data dictionary status:",
    data_dictionary_status,
)

print(
    "Portfolio Return Dataset marker count:",
    marker_count,
)

Data dictionary status: ALREADY EXISTS
Portfolio Return Dataset marker count: 1


In [ ]:

# Idempotently update data-source documentation

DATA_SOURCE_PATH = (
    PROJECT_ROOT / "docs" / "data-source.md"
)

SECTION_MARKER = "## Portfolio and Benchmark Data Usage"


if not DATA_SOURCE_PATH.exists():
    raise FileNotFoundError(
        f"Data-source documentation not found: {DATA_SOURCE_PATH}"
    )


existing_data_source = DATA_SOURCE_PATH.read_text(
    encoding="utf-8"
)


portfolio_data_source_section = """
## Portfolio and Benchmark Data Usage

### Portfolio constituent data

The default portfolio consists of three equities:

- HPG
- FPT
- MWG

Validated processed datasets are stored at:

- `data/processed/HPG_clean.csv`
- `data/processed/FPT_clean.csv`
- `data/processed/MWG_clean.csv`

These three assets are the constituents used to construct the default
equal-weight portfolio.

### Derived portfolio data

The canonical derived portfolio-return dataset is stored at:

`data/processed/portfolio_returns.csv`

A compact review sample is stored at:

`data/sample/portfolio_returns_sample.csv`

The canonical portfolio-return dataset contains 1,637 valid daily
return observations from 2020-01-03 through 2026-07-28.

The primary return series used by the downstream Value at Risk
pipeline is:

`portfolio_simple_return`

This series is derived from the validated HPG, FPT, and MWG return
series and their portfolio weights.

### VN-Index benchmark

The raw VN-Index dataset is stored at:

`data/raw/VNINDEX.csv`

It contains 1,638 observations covering 2020-01-02 through
2026-07-28.

VN-Index is used only as a market benchmark and is not a constituent
of the default portfolio.

Its portfolio weight is therefore zero.

The raw VN-Index timestamp contains a `07:00:00` time component.
For daily trading-calendar comparison, this timestamp is normalized
to midnight.

This normalization removes only the time-of-day component and does
not change the trading date.

After trading-date normalization, the VN-Index and portfolio price
calendars contain 1,638 common dates and are fully aligned.

At this stage, no processed:

`data/processed/VNINDEX_clean.csv`

dataset has been established.

Therefore:

`data/raw/VNINDEX.csv`

must not be treated as model-ready processed data.

### Data lineage rule

Downstream portfolio-risk and Value at Risk components must use the
canonical:

`portfolio_simple_return`

series from:

`data/processed/portfolio_returns.csv`

Portfolio returns must not be reconstructed from the raw VN-Index
dataset.

VN-Index remains a separate benchmark/context series unless the
project methodology is explicitly changed and revalidated.
""".strip()


# Append only when the section does not already exist

if SECTION_MARKER not in existing_data_source:
    updated_data_source = (
        existing_data_source.rstrip()
        + "\n\n"
        + portfolio_data_source_section
        + "\n"
    )

    DATA_SOURCE_PATH.write_text(
        updated_data_source,
        encoding="utf-8",
    )

    data_source_status = "CREATED"

else:
    data_source_status = "ALREADY EXISTS"


 
# Post-run validation
 

data_source_after_run = DATA_SOURCE_PATH.read_text(
    encoding="utf-8"
)

marker_count = data_source_after_run.count(
    SECTION_MARKER
)


if marker_count != 1:
    raise ValueError(
        "Unexpected Portfolio and Benchmark Data Usage marker count: "
        f"{marker_count}"
    )


print(
    "Data source status:",
    data_source_status,
)

print(
    "Portfolio and Benchmark Data Usage marker count:",
    marker_count,
)

Data source status: ALREADY EXISTS
Portfolio and Benchmark Data Usage marker count: 1
